In [ ]:
!pip install torch 

: 

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import WeightedRandomSampler, DataLoader

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Code").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))    

DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "peer"
RESULTS_ROOT = PROJECT_ROOT / "Code" / "results" / "esm2"
CHECKPOINTS_ROOT = PROJECT_ROOT / "checkpoints"

print("Project root:", PROJECT_ROOT)
print("Data root exists:", DATA_ROOT.exists())
print("Results root exists :", RESULTS_ROOT.exists())
print("Checkpoints root:", CHECKPOINTS_ROOT)

: 

#### Dataset Loader

In [ ]:
def load_peer_split(dataset: str, split: str) -> pd.DataFrame:
    """
    Load the peer dataset split.

    Args:
        dataset (str): The name of the dataset.
        split (str): The name of the split (train, val, test).

    Returns:
        pd.DataFrame: The loaded dataset split.
    """
    path = DATA_ROOT / dataset / f"{split}.csv"
    if not path.exists():
        raise FileNotFoundError(f"File {path} does not exist.")
    
    df = pd.read_csv(path)
    df["split"] = split
    df["dataset"] = dataset

    return df

solubility = pd.concat(
    [
        load_peer_split("solubility", "train"),
        load_peer_split("solubility", "valid"),
        load_peer_split("solubility", "test"),
    ],
    ignore_index = True,
)

localization = pd.concat(
    [
        load_peer_split("localization", "train"),
        load_peer_split("localization", "valid"),
        load_peer_split("localization", "test"),
    ],
    ignore_index = True,
)
print("Solubility shape:", solubility.shape)
print("Localization shape:", localization.shape)

display(solubility.head())
display(localization.head())

In [ ]:
print("solubility columns:", solubility.columns.tolist())
print("localization columns:", localization.columns.tolist())

In [ ]:
SEQUENCE_COLUMN = "sequence"
LABEL_COLUMN = "label"

#### Quality Check

In [ ]:
def quality_report(df: pd.DataFrame, sequence_column: str, label_column: str) -> pd.Series:
    return pd.Series(
        {
            "total_samples": len(df),
            "missing_sequences" : df[sequence_column].isna().sum(),
            "missing_labels": df[label_column].isna().sum(),
            "duplicate_sequences": df[sequence_column].duplicated().sum(),
            "unique_sequences": df[sequence_column].nunique(),
            "unique_labels": df[label_column].nunique(),
            "minimum_length": df[sequence_column].str.len().min(),
            "maximum_length": df[sequence_column].str.len().max(),
            "mean_length": df[sequence_column].str.len().mean(),
            "maximum_length": df[sequence_column].str.len().max(),
        }
    )

display(
    quality_report(
        solubility,
        SEQUENCE_COLUMN,
        LABEL_COLUMN
    ).to_frame("solubility")
)

display(
    quality_report(
        localization,
        SEQUENCE_COLUMN,
        LABEL_COLUMN
    ).to_frame("localization")
)



In [ ]:
def class_distribution(
        df: pd.DataFrame,
        label_column: str,
        dataset_name: str
) -> pd.DataFrame:
    """
    Calculate the class distribution for a given dataset.

    Args:
        df (pd.DataFrame): The input DataFrame containing the dataset.
        label_column (str): The name of the column containing the labels.
        dataset_name (str): The name of the dataset.

    Returns:
        pd.DataFrame: A DataFrame containing the class distribution.
    """
    counts = (
        df.groupby(["split", label_column])
        .size().rename("count").reset_index()
    )

    counts["proportion"] = counts.groupby("split")["count"].transform(
        lambda values: values / values.sum()
    )

    counts["dataset"] = dataset_name
    return counts

sol_distribution = class_distribution(
    solubility,
    LABEL_COLUMN,
    "solubility",
)

loc_distribution = class_distribution(
    localization,
    LABEL_COLUMN,
    "localization",
)

display(sol_distribution)
display(loc_distribution)

In [ ]:
train_sol_counts = (
    solubility.loc[solubility["split"] == "train",  LABEL_COLUMN]
    .value_counts()
    .sort_index()
)

train_sol_counts.plot(kind="bar")
plt.title("Solubility Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Sequences")
plt.tight_layout()
plt.show()

In [ ]:
train_loc_counts = (
    localization.loc[localization["split"] == "train",  LABEL_COLUMN]
    .value_counts()
    .sort_index()
)

train_loc_counts.plot(kind="bar")
plt.title("Localization Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Sequences")
plt.tight_layout()
plt.show()

#### Sequence length analysis

In [ ]:
for dataframe in (solubility, localization):
    dataframe["sequence_length"] = dataframe[SEQUENCE_COLUMN].str.len()

In [ ]:
solubility["sequence_length"].plot(
    kind="hist",
    bins=50,
)
plt.title("Solubility Sequence-Length Distribution")
plt.xlabel("Sequence Length")
plt.ylabel("Number of Sequences")
plt.tight_layout()
plt.show()

In [ ]:
solubility.boxplot(
    column="sequence_length",
    by=LABEL_COLUMN,
)
plt.title("Solubility Sequence Length by Class")
plt.suptitle("")
plt.xlabel("Class")
plt.ylabel("Sequence Length")
plt.tight_layout()
plt.show()

#### Class Weights and Wighted Sampling

In [ ]:
def calculate_weights(
    labels: pd.Series,
) -> tuple[np.ndarray, torch.Tensor]:
    classes = np.sort(labels.unique())

    weights = compute_class_weight(
        class_weight="balanced",
        classes = classes,
        y = labels.to_numpy(),
    )
    return classes, torch.tensor(weights, dtype=torch.float32)

In [ ]:
sol_train_labels = solubility.loc[
    solubility["split"] == "train",
    LABEL_COLUMN,
]

sol_classes, sol_class_weights = calculate_weights(sol_train_labels)

sol_weight_table = pd.DataFrame(
    {
        "class": sol_classes,
        "class_weight": sol_class_weights.numpy(),
        "count": [
            (sol_train_labels == label).sum()
            for label in sol_classes
        ],
    }
)

display(sol_weight_table)

In [ ]:
loc_train_labels = localization.loc[
    localization["split"] == "train",
    LABEL_COLUMN,
]

loc_classes, loc_class_weights = calculate_weights(loc_train_labels)

loc_weight_table = pd.DataFrame(
    {
        "class": loc_classes,
        "class_weight": loc_class_weights.numpy(),
        "count": [
            (loc_train_labels == label).sum()
            for label in loc_classes
        ],
    }
)

display(loc_weight_table)

In [ ]:
criterion = torch.nn.CrossEntropyLoss(
    weight=sol_class_weights.to(device)
    )

#### Create ampler weights

In [ ]:
def create_sample_weights(labels: pd.Series) -> torch.Tensor:
    class_counts = labels.value_counts().sort_index()
    inverse_frequency = 1.0 / class_counts

    sample_weights = labels.map(inverse_frequency).to_numpy()

    return torch.tensor(
        sample_weights,
        dtype=torch.double,
    )


sol_sample_weights = create_sample_weights(sol_train_labels)

sol_sampler = WeightedRandomSampler(
    weights=sol_sample_weights,
    num_samples=len(sol_sample_weights),
    replacement=True,
)

print("Number of sampler weights:", len(sol_sample_weights))
print("First five weights:", sol_sample_weights[:5])

#### Training History Dashboard

In [ ]:
history_files = sorted(
    RESULTS_ROOT.rglob("history.json")
)

print(f"Found {len(history_files)} history files.")

for path in history_files[:10]:
    print(path.relative_to(PROJECT_ROOT))

#### Benchmark Table

In [ ]:
def load_benchmark_record(path: Path) -> dict:
    with open(path, "r") as file:
        history = json.load(file)

    hyperparameters = history.get("hyperparameters", {})
    summary = history.get("summary", {})

    return {
        "run_path": str(path.parent.relative_to(PROJECT_ROOT)),
        "dataset": hyperparameters.get("dataset"),
        "classifier_head": hyperparameters.get("classifier_head"),
        "fine_tuning_strategy": hyperparameters.get(
            "fine_tuning_strategy"
        ),
        "learning_rate": hyperparameters.get("learning_rate"),
        "esm_learning_rate": hyperparameters.get(
            "esm_learning_rate"
        ),
        "best_epoch": summary.get("best_epoch"),
        "epochs_trained": summary.get("epochs_trained"),
        "best_val_loss": summary.get("best_val_loss"),
        "best_val_accuracy": summary.get("best_val_accuracy"),
        "best_val_f1": summary.get("best_val_f1"),
        "best_val_precision": summary.get(
            "best_val_precision"
        ),
        "best_val_recall": summary.get("best_val_recall"),
    }


benchmark_df = pd.DataFrame(
    load_benchmark_record(path)
    for path in history_files
)

benchmark_df = benchmark_df.sort_values(
    ["dataset", "best_val_f1"],
    ascending=[True, False],
)

display(benchmark_df)

#### Learning Curve

In [ ]:
selected_history_path = history_files[0]

with open(selected_history_path, "r") as file:
    selected_history = json.load(file)

epochs_df = pd.DataFrame(selected_history["epochs"])
display(epochs_df.head())

In [ ]:
plt.plot(
    epochs_df["epoch"],
    epochs_df["train_loss"],
    label="Train",
)
plt.plot(
    epochs_df["epoch"],
    epochs_df["val_loss"],
    label="Validation",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(
    epochs_df["epoch"],
    epochs_df["train_accuracy"],
    label="Train",
)
plt.plot(
    epochs_df["epoch"],
    epochs_df["val_accuracy"],
    label="Validation",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()